# IBM Fraud Precursor EDA

This notebook inspects the dataset and measures whether backward windows of length 5, 10, and 20 are feasible. Run it from the project root.

In [ ]:
from pathlib import Path
import os
import sys
project_root = Path.cwd()
if not (project_root / 'src').exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from src.config import Config
from src.data_loader import load_transactions
from src.preprocessing import clean_transactions
from src.reporting import dataset_summary

config_path = Path(os.environ.get('FRAUD_CONFIG', 'config.yaml'))
if not config_path.is_absolute():
    config_path = project_root / config_path
config = Config.from_yaml(config_path)
raw = load_transactions(config.dataset_path, config.chunksize)
df = clean_transactions(raw, config.user_column, config.target_column)

In [ ]:
display(pd.DataFrame({'dtype': df.dtypes.astype(str), 'missing': df.isna().sum()}))
display(dataset_summary(df, config.user_column, config.target_column, config.sequence_length))

In [ ]:
per_user = df.groupby(config.user_column).size()
fraud_per_user = df.groupby(config.user_column)[config.target_column].sum()
display(pd.DataFrame({
    'users_with_no_fraud': [(fraud_per_user == 0).sum()],
    'users_with_fraud': [(fraud_per_user > 0).sum()],
    'total_users': [len(per_user)],
}))
feasibility = []
position_in_user = df.groupby(config.user_column).cumcount()
for n in (5, 10, 20):
    eligible_targets = df.loc[position_in_user >= n, config.target_column]
    feasibility.append({
        'sequence_length': n,
        'eligible_users': int((per_user >= n + 1).sum()),
        'total_windows': int(len(eligible_targets)),
        'fraud_preceding_windows': int(eligible_targets.sum()),
        'normal_preceding_windows': int(len(eligible_targets) - eligible_targets.sum()),
    })
display(pd.DataFrame(feasibility))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
sns.histplot(per_user, bins=50, ax=axes[0, 0]).set(title='Transactions per user')
sns.histplot(fraud_per_user, bins=50, ax=axes[0, 1]).set(title='Frauds per user')
sns.countplot(data=df, x=config.target_column, ax=axes[0, 2]).set(title='Fraud class')
sns.histplot(df['Amount'], bins=80, ax=axes[1, 0]).set(title='Amount')
sns.histplot(data=df[df[config.target_column] == 1], x='Amount', bins=80, ax=axes[1, 1]).set(title='Fraud amount')
if 'Use Chip' in df:
    sns.countplot(data=df, y='Use Chip', hue=config.target_column, ax=axes[1, 2]).set(title='Use Chip')
plt.tight_layout()

In [ ]:
if 'MCC' in df:
    display(df.groupby('MCC')[config.target_column].agg(['count', 'mean']).sort_values('count', ascending=False).head(30))
display(df.assign(hour=df.datetime.dt.hour, day_of_week=df.datetime.dt.day_name()).groupby(['day_of_week', 'hour'])[config.target_column].agg(['count', 'mean']))